# Segundo Trabalho — Reconhecimento de Padrões

Notebook para as questões do **segundo** trabalho. O relatório em LaTeX será montado depois, em `documents/trabalho2/`.

**Dependências:** `numpy`, `matplotlib`.

## Questão 1 — Definição das classes

Problema de classificação com **duas classes** em $\mathbb{R}^2$, cada uma modelada por uma **Gaussiana bidimensional** com a **mesma** matriz de covariância $\Sigma$:

- **Classe $C_1$:** $\mathbf{x} \mid C_1 \sim \mathcal{N}(\boldsymbol{\mu}_1, \boldsymbol{\Sigma})$, com $\boldsymbol{\mu}_1 = [0,0]^\top$ e
  $$\boldsymbol{\Sigma} = \begin{bmatrix} 1 & 0{,}6 \\ 0{,}6 & 1 \end{bmatrix}.$$
- **Classe $C_2$:** $\mathbf{x} \mid C_2 \sim \mathcal{N}(\boldsymbol{\mu}_2, \boldsymbol{\Sigma})$, com $\boldsymbol{\mu}_2 = [2,0]^\top$ e a mesma $\boldsymbol{\Sigma}$.

Cada classe fica assim descrita por **dois parâmetros** (vetor média em $\mathbb{R}^2$), além da covariância **compartilhada** $\boldsymbol{\Sigma}$.

## Questão 2 — Geração de dados e gráfico com a fronteira do 1º trabalho

Conforme o enunciado:

1. Sortear um inteiro em $[0,500]$ como semente (reproduzível a partir de uma semente-mestre fixa abaixo).
2. Gerar **1000** amostras de treino para $C_1$ e **1000** para $C_2$.
3. Plotar os pontos e a **superfície de separação obtida no primeiro trabalho** para esta mesma configuração $(\boldsymbol{\mu}_1, \boldsymbol{\mu}_2, \boldsymbol{\Sigma})$ — isto é, a **fronteira ótima de Bayes** com classes equiprováveis e mesma covariância:
   $$h(\mathbf{x}) = \boldsymbol{w}^\top \mathbf{x} + b = 0,\qquad
   \boldsymbol{w} = \boldsymbol{\Sigma}^{-1}(\boldsymbol{\mu}_2-\boldsymbol{\mu}_1),\quad
   b = -\tfrac12\bigl(\boldsymbol{\mu}_2^\top\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}_2 - \boldsymbol{\mu}_1^\top\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}_1\bigr).$$

Para $\boldsymbol{\mu}_1=\mathbf{0}$, $\boldsymbol{\mu}_2=(2,0)^\top$ e a $\boldsymbol{\Sigma}$ acima, isso coincide com a *Questão 8* do primeiro trabalho: $h(\mathbf{x}) = 3{,}125\,x_1 - 1{,}875\,x_2 - 3{,}125$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Parâmetros da Questão 1 ---
mu1 = np.array([0.0, 0.0])
mu2 = np.array([2.0, 0.0])
Sigma = np.array([[1.0, 0.6], [0.6, 1.0]])

# Fronteira de Bayes (mesma derivação do 1º trabalho; priors iguais)
Sigma_inv = np.linalg.inv(Sigma)
w = Sigma_inv @ (mu2 - mu1)
b = -0.5 * (mu2 @ Sigma_inv @ mu2 - mu1 @ Sigma_inv @ mu1)

# --- Questão 2: semente em [0, 500] e amostras ---
MASTER_SEED = 2027  # fixa a reprodutibilidade do sorteio do enunciado
seed_train = int(np.random.default_rng(MASTER_SEED).integers(0, 501))
rng = np.random.default_rng(seed_train)

n = 1000
X1 = rng.multivariate_normal(mu1, Sigma, size=n)
X2 = rng.multivariate_normal(mu2, Sigma, size=n)

FIG_DIR = Path("../documents/trabalho2/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

x_all = np.vstack([X1, X2])
pad = 0.8
x_min, y_min = x_all.min(axis=0) - pad
x_max, y_max = x_all.max(axis=0) + pad

plt.figure(figsize=(7.0, 6.2))
plt.plot(X1[:, 0], X1[:, 1], "r+", markersize=4, label="$C_1$")
plt.plot(X2[:, 0], X2[:, 1], "b.", markersize=4, label="$C_2$")

x1_line = np.linspace(x_min, x_max, 400)
if abs(w[1]) > 1e-12:
    x2_line = -(w[0] * x1_line + b) / w[1]
    m = (x2_line >= y_min) & (x2_line <= y_max)
    plt.plot(x1_line[m], x2_line[m], "g.", markersize=2, label="Bayes (1º trabalho)")
else:
    xv = np.full(300, -b / w[0])
    yv = np.linspace(y_min, y_max, 300)
    plt.plot(xv, yv, "g.", markersize=2, label="Bayes (1º trabalho)")

plt.xlabel("Parâmetro 1")
plt.ylabel("Parâmetro 2")
plt.title("Questão 2 — Treino ($n=1000$/classe) e fronteira de Bayes")
plt.grid(True, alpha=0.35)
plt.axis([x_min, x_max, y_min, y_max])
plt.gca().set_aspect("equal", adjustable="box")
plt.legend(loc="best")

out = FIG_DIR / "questao2_treino_fronteira_bayes_t1.png"
plt.savefig(out, dpi=180, bbox_inches="tight")
plt.show()

print(f"Semente sorteada (intervalo [0, 500]): {seed_train}")
print(f"Fronteira: ({w[0]:.4f})*x1 + ({w[1]:.4f})*x2 + ({b:.4f}) = 0")
print(f"Figura salva em: {out.resolve()}")